# PyCAM-SIMA Dask checkpoint fan-out

This standalone Notebook demonstrates the task-oriented execution mode. It does not start `NotebookSession` and does not keep an MPI job alive behind a socket. Dask first submits one 24-rank base PBS/MPI segment, retains its immutable Python-owned checkpoint as a Future, verifies an exact `+1 K` edit in a zero-step checkpoint, and then continues three independent experiments.

## 1. Execution model

```text
Jupyter + local Dask Client
           │
           ├── base PBS job: 24 MPI ranks × 10 steps
           │                    │
           │                    └── immutable checkpoint Future
           │                                  │
           ├──────────────────────────────────┼── control:    5 steps
           ├──────────────────────────────────└── no-kessler: 5 steps
           └── warm-initial: +1 K, 0 steps
                         │ exact edit check
                         └── warm: 5 steps
```

The base process exits after writing its checkpoint. Each segment restores private NumPy arrays and creates a new `MPI.COMM_WORLD`; this is checkpoint/restart fan-out, not operating-system `fork()`. The zero-step `warm-initial` checkpoint separates verification of the requested edit from the later model response.

## 2. Configure one experiment

Run this cell again to create a fresh timestamp before repeating the experiment. Existing branch directories are deliberately never overwritten.

In [ ]:
from datetime import datetime
from pathlib import Path
import os
import shutil

import numpy as np
import pycam_sima
from dask.distributed import Client
from netCDF4 import Dataset
from pycam_sima import BranchSpec, DaskExperimentClient, FieldEdit

repo = Path('/glade/work/ruitong/pycam-sima')
scratch = Path(os.environ.get('SCRATCH', '/glade/derecho/scratch/ruitong'))
config_path = repo / 'configs/fkessler_model.yaml'
reference_atm_in = (
    repo / 'reference/cases/FKESSLER_ne3pg3_gnu_24x50/CaseDocs/atm_in'
)

stamp = datetime.now().strftime('%Y%m%d-%H%M%S')
experiment_root = scratch / 'pycam-sima/dask_notebook_trials' / f'fanout-{stamp}'
initial_run_dir = experiment_root / 'initial-run'
run_root = experiment_root / 'branches'
initial_run_dir.mkdir(parents=True, exist_ok=False)
shutil.copy2(reference_atm_in, initial_run_dir / 'atm_in')

print('pycam_sima', pycam_sima.__version__)
print('experiment root:', experiment_root)

## 3. Create the Dask controller

The three local Dask workers only orchestrate PBS submissions. Every actual model segment still runs on 24 MPI ranks.

In [ ]:
if 'client' in globals():
    client.close()

client = Client(
    processes=False,
    n_workers=3,
    threads_per_worker=1,
    dashboard_address=None,
)
experiments = DaskExperimentClient(
    client,
    config=config_path,
    initial_run_dir=initial_run_dir,
    run_root=run_root,
    python_executable=repo / '.venv/bin/python',
)
client

## 4. Submit the common base and the zero-step warm edit

This first stage submits two PBS jobs. The base runs 10 steps. `warm-initial` restores that exact state, adds `1 K` to `air_temperature`, runs zero model steps, and writes another checkpoint.

In [ ]:
base = experiments.submit_base(
    BranchSpec('base', steps=10)
)

warm_initial = experiments.submit_branch(
    base,
    BranchSpec(
        'warm-initial',
        steps=0,
        field_edits=(
            FieldEdit('air_temperature', 'add', 1.0),
        ),
    ),
)

initial_summaries = experiments.summaries({
    'base': base,
    'warm-initial': warm_initial,
})
initial_summaries

## 5. Verify the edit before model evolution

Both checkpoints are at model step 10. This comparison therefore tests only `FieldEdit('air_temperature', 'add', 1.0)` across all 24 ranks. Every edited array must match NumPy's elementwise `base + 1.0` operation bit for bit. Recomputing `(base + 1.0) - base` performs another floating-point operation, so that diagnostic is required to equal `1 K` only within one spacing of the largest base value.

In [ ]:
def checkpoint_field(summary_map, branch, field, rank=0):
    checkpoint_file = (
        Path(summary_map[branch]['checkpoint_dir'])
        / f'rank-{rank:03d}.npz'
    )
    with np.load(checkpoint_file, allow_pickle=False) as arrays:
        return arrays[field].copy()

base_checkpoint = Path(initial_summaries['base']['checkpoint_dir'])
rank_count = len(tuple(base_checkpoint.glob('rank-*.npz')))
base_temperatures = [
    checkpoint_field(initial_summaries, 'base', 'air_temperature', rank)
    for rank in range(rank_count)
]
warm_initial_temperatures = [
    checkpoint_field(
        initial_summaries, 'warm-initial', 'air_temperature', rank
    )
    for rank in range(rank_count)
]
initial_differences = [
    warm - base
    for base, warm in zip(base_temperatures, warm_initial_temperatures)
]
exact_numpy_edit = all(
    np.array_equal(warm, np.add(base, 1.0))
    for base, warm in zip(base_temperatures, warm_initial_temperatures)
)
maximum_roundoff_from_1K = float(
    max(np.abs(delta - 1.0).max() for delta in initial_differences)
)
roundoff_tolerance = float(
    max(np.spacing(np.abs(base).max()) for base in base_temperatures)
)
difference_within_roundoff = (
    maximum_roundoff_from_1K <= roundoff_tolerance
)

assert exact_numpy_edit
assert difference_within_roundoff
sample_count = sum(delta.size for delta in initial_differences)
{
    'ranks': rank_count,
    'rank_local_shape': initial_differences[0].shape,
    'difference_min': float(min(delta.min() for delta in initial_differences)),
    'difference_max': float(max(delta.max() for delta in initial_differences)),
    'difference_mean': float(
        sum(delta.sum() for delta in initial_differences) / sample_count
    ),
    'exact_numpy_add_1K': exact_numpy_edit,
    'maximum_roundoff_from_1K': maximum_roundoff_from_1K,
    'roundoff_tolerance': roundoff_tolerance,
    'difference_within_roundoff': difference_within_roundoff,
}

## 6. Continue the three five-step experiments

`control` and `no-kessler` continue directly from the base checkpoint. `warm` continues from the already verified `warm-initial` checkpoint without applying a second edit. This stage submits three additional PBS jobs.

In [ ]:
branches = experiments.fork(
    base,
    (
        BranchSpec('control', steps=5),
        BranchSpec(
            'no-kessler',
            steps=5,
            disable_schemes=('kessler',),
        ),
    ),
)
branches['warm'] = experiments.submit_branch(
    warm_initial,
    BranchSpec('warm', steps=5),
)

summaries = experiments.summaries(branches)
summaries

## 7. What the summary tells you

All three final branches report `step=15` and 15 history samples. `control` and `no-kessler` have `parent_branch='base'`; `warm` has `parent_branch='warm-initial'`. The summary also contains each PBS job ID, run/history/checkpoint/log paths, and serialized checkpoint size without downloading the full checkpoint Future.

In [ ]:
[
    {
        'branch': name,
        'step': summary['step'],
        'history_samples': summary['history_samples'],
        'pbs_job_id': summary['pbs_job_id'],
        'checkpoint_GiB': summary['snapshot_nbytes'] / 1024**3,
        'history_dir': summary['history_dir'],
        'log_path': summary['log_path'],
    }
    for name, summary in summaries.items()
]

## 8. Compare the temperature after five model steps

Every branch checkpoint contains the complete 214-field StatePool for all 24 ranks. Unlike the exact pre-run edit, the warm-control difference after five nonlinear model steps is expected to vary around `1 K`. The deviation from `1 K`, rather than the total warm-control difference, measures the subsequent model response.

In [ ]:
control_temperature = checkpoint_field(
    summaries, 'control', 'air_temperature'
)
warm_temperature = checkpoint_field(
    summaries, 'warm', 'air_temperature'
)
temperature_difference = warm_temperature - control_temperature

{
    'rank': 0,
    'shape': control_temperature.shape,
    'difference_min': float(temperature_difference.min()),
    'difference_max': float(temperature_difference.max()),
    'difference_mean': float(temperature_difference.mean()),
    'maximum_deviation_from_1K': float(
        np.abs(temperature_difference - 1.0).max()
    ),
    'bitwise_identical': bool(np.array_equal(control_temperature, warm_temperature)),
}

## 9. Inspect a global field after every model step

Each branch history directory inherits the 10 base timestamps and adds 5 branch timestamps. These NetCDF files contain the 26 configured global diagnostics. Unlike the final rank-local checkpoint, this gives one global field value at every completed model step.

In [ ]:
def history_statistics(branch, variable):
    summary = summaries[branch]
    history_dir = summary.get(
        'history_dir',
        Path(summary['checkpoint_dir']).parent / 'history',
    )
    files = sorted(Path(history_dir).glob('*.nc'))
    records = []
    for path in files:
        with Dataset(path) as dataset:
            values = np.asarray(dataset[variable][0])
            records.append({
                'step': int(dataset['nsteph'][0]),
                'file': path.name,
                'minimum': float(values.min()),
                'maximum': float(values.max()),
                'mean': float(values.mean()),
            })
    return records

control_rain_by_step = history_statistics('control', 'RAINQM')
no_kessler_rain_by_step = history_statistics('no-kessler', 'RAINQM')

{
    'control_last': control_rain_by_step[-1],
    'no_kessler_last': no_kessler_rain_by_step[-1],
    'control_all_steps': control_rain_by_step,
}

## 10. Observation boundary

This Dask mode observes all 214 StatePool fields at segment boundaries and the 26 NetCDF diagnostics after every step. It does not pause a running branch inside a step. To branch or inspect all fields at every step, submit chained one-step segments (`steps=1`) so that each task boundary produces another complete checkpoint.

In [ ]:
client.close()
print('Dask client closed; PBS results remain under', experiment_root)